# twinfit: five minutes, an atomic clock at five probe powers

Find what your model is missing. twinfit fits one model to several measured curves, reads the residuals against synthetic curves drawn from that fit, and tells you where the model misses, what the miss looks like and what it does to the parameters the curves share.

The five minutes of the package's README, in a notebook: its words and its commands in their order. Each command runs through the shell, `!`, so the notebook needs nothing but `pip install twinfit`.

In [ ]:
!pip install twinfit
!python -m twinfit example clock clock
!python -m twinfit loop clock

In [ ]:
print(open("clock/loop/model_0.md").read())

The second line writes a project into `clock`: five resonances of a clock transition against the detuning of the probe, at 2 to 10 uW, with a first model of one Lorentzian line of one width and one centre. The powers were read by a photodiode calibrated within 2 per cent, which `clock/description.toml` holds as an ansatz under `power scale`. The truth of the project is in the gallery, `python -m twinfit guide examples`.

The loop fits the five curves at once, reads their residuals against 30 synthetic sets drawn from the fit, and writes `clock/loop/model_0.md`. Its section "Where to search" gives the first answer to each question. The largest excess that names a part of the curves is here the question `changes along`:

What to expect:

```
| changes along | which part of the residuals changes from curve to curve along a setting, read on the residuals themselves | the asymmetry along power | 18388.1 |
```

The report names no term. The physics does: an asymmetry that grows with the power of the probe is a centre that moves with the power, a light shift, which the first model lacks. Write it into `clock/model.py`. The centre at a power P becomes the clock frequency plus the light shift times P:

In [ ]:
%%writefile clock/model.py
"""A model of one's own: the centre of the line moves with the power of the probe, the light shift, from the frequency of
the clock at no power, which the model then measures."""
import numpy as np

SHARED = ("width", "clock frequency", "light shift")      # the full width, in Hz, the centre at no power less the nominal frequency, in Hz, and its shift, in Hz for each microwatt
OWN = ("height", "background")                           # of each record
START_SHARED = [1.0, 0.0, 0.0]
BOUNDS_SHARED = ([0.05, -3.0, -1.0], [10.0, 3.0, 1.0])
HELD = {"power scale": 1.0}                              # the powers read times this, calibrated within 2 per cent, which description.toml holds
NAME = "one Lorentzian line whose centre moves with the power"
EQUATION = r"$\frac{h}{1 + (2\,(x - \nu_0 - \delta\,P)/w)^2} + b$"          # the model in LaTeX, which the visual log typesets
LETTERS = {"h": "the height", r"\nu_0": "the clock frequency", r"\delta": "the light shift for each microwatt", "P": "the power", "w": "the width", "b": "the background"}          # each letter of EQUATION, which the visual log writes under it


def model(x, shared, own, design):
    centre = shared[1] + shared[2] * design["power"] * HELD["power scale"]
    return own[0] / (1.0 + (2.0 * (x - centre) / shared[0]) ** 2) + own[1]


def start_own(record):
    base = float(np.median(record.y))
    return [float(record.y.max() - base), base]

Run `python -m twinfit loop clock` again. The report finds nothing beyond the twin, and `clock/loop/best.md` reports:

In [ ]:
!python -m twinfit loop clock

In [ ]:
print(open("clock/loop/best.md").read())

What to expect:

```
- clock frequency: 0.12000 ± 0.00038 of the fit
- light shift: 0.025051 ± 0.000058 of the fit ± 0.00050 of theory
```

The uncertainty from theory is the 2 per cent of the calibration, carried through the fit to the light shift and not to the clock frequency. Compare both with the truth in the gallery.

On the first model that `example clock` writes, whose centre is a shared parameter, a term file `clock/terms/01.py` with `term = mechanisms.along("centre", "power")` finds the same light shift, as the line of the centre along the power that `best.md` derives from the size of the term, and the same clock frequency at power 0 to the digit, but it reads no calibration, so its bar carries no form and the held values move nothing. Where the physics gives the law, write it in `model.py`.

Each run takes between ten seconds and a minute on a laptop. The loop writes `clock/loop/` again at each run, so copy it to keep a report. `best.md` may ask for `--sets 500` before a value is quoted. `python -m twinfit guide tutorial` prints the tutorial and `python -m twinfit guide examples` the gallery.

`python -m twinfit guide rubidium` shows a real case: recorded traces of the two photon line of rubidium, as plots, with what the package reads on them.